## Libraries

In [35]:
import os

In [36]:
run = 'local' #local or cloud
os.environ["APP_ENV"] = run

In [37]:
import sys
sys.path.append('..')

In [38]:
from src.core_grass import*
%matplotlib inline
#%load_ext autoreload
#%autoreload 2

In [39]:
import subprocess
result = subprocess.run(["grass", "--config", "python_path"],
                        check=True, text=True, capture_output=True)
sys.path.append(result.stdout.strip())

# import GRASS Python packages
from collections import defaultdict
import grass.script as gs
import grass.jupyter as gj

# Parameters

In [40]:
#set project variables
Project_Area = 'glen_prov'
projection = '2272' #set to None if you want to use the DEM's original projection
res = '1m'

## Set variables analysis
source = 'wbd' #wbd or user-defined
data_scale = 'HUC12' #other options HUC12, HUC10, HUC8 OR the field name of the source data to be split <-- case sensitive
select_data = '020402020602' #     # Value within the filter_field used for data selection 
to_headwaters = False #False if only interested in local huc area
input_shp_name = None #None or must be populated populated with a shapefile polygons if data_scale != HUCXX
buffer = 2000
auto_delete = False

In [41]:
if data_scale.find('HUC') >= 0:
    huc_level = re.findall("[0-9]+",data_scale)[0]
    if huc_level == str(len(select_data)):
        huc_digys = huc_level
    else:
        huc_digys = huc_level
        select_data = select_data[:int(huc_digys)]

In [42]:
## Paths:
parent_dir = pl.Path(os.getcwd()).parent/'data'
grass_data_dir = pl.Path(os.getcwd()).parent/'grassdata'
current_dir = pl.Path(os.getcwd())
outputs_dir =parent_dir/'Rasters'/Project_Area/'Outputs'
vector_dir = parent_dir/'Vectors'/Project_Area
raster_dir = parent_dir/'Rasters'/Project_Area
huc_path = vector_dir/'{}_aoi.geojson'.format(Project_Area)
git_data_repo = current_dir/'data_repository'
if input_shp_name:
    input_shp = vector_dir/input_shp_name
directory = outputs_dir

### Get area of interest

In [43]:
#read from to vector path from notebook 1
huc_path = vector_dir/'{0}_{1}_aoi.geojson'.format(Project_Area,select_data)
aoi = gpd.read_file(str(huc_path),driver="GeoJSON")

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


### Start new grass session

In [44]:
# define GRASS data settings (adapt to your needs)
location = Project_Area+'_'+projection

In [45]:
# Create a project if it doesn't exist
if os.path.exists(grass_data_dir/location):
    pass
else:
    if len(projection) > 4:
        wkt_text = pyproj.CRS.from_authority("esri", int(projection)).to_wkt()
        with open(vector_dir/'{}_prj.wkt'.format(Project_Area), 'w') as t:
            t.write(wkt_text)
        gs.create_project(path=grass_data_dir,
                      name=location,
                      wkt=vector_dir/'{}_prj.wkt'.format(Project_Area))
    else:
        gs.create_project(path=grass_data_dir,
                  name=location,
                  epsg=projection)

In [46]:
# Start GRASS in default project mapset
session = gj.init(grass_data_dir/location)

#### Review existing files

In [47]:
#list all grass files
grass_maps = list_existing_grass(print_it=True)

Available vector maps:
aoi@PERMANENT
aoi_buffer@PERMANENT
outlet_020402020602@PERMANENT
outlet_020402020602_snap@PERMANENT
stream_020402020602@PERMANENT
tmp_bsn@PERMANENT
tmp_bsn_buffer@PERMANENT
v_basins_020402020602@PERMANENT
v_carve_020402020602@PERMANENT

Available raster maps:
MASK@PERMANENT
accum_020402020602@PERMANENT
dem_020402020602@PERMANENT
dem_020402020602_b@PERMANENT
drain_dir_020402020602@PERMANENT
r_basins_020402020602@PERMANENT
r_streams_020402020602@PERMANENT
r_twi_020402020602@PERMANENT
subbasins_020402020602@PERMANENT


In [48]:
basin_list = aoi[data_scale.lower()].to_list()
basins = 'aoi'
gs.parse_command('v.import', input= huc_path,  output= basins)
print('added aoi to grass')

added aoi to grass


In [49]:
#set variables
carved = True
carve_lines = vector_dir/'carve.shp'
outlet_point = vector_dir/f'v_{select_data}_outlet.shp'

In [50]:
#standard variables
threshold = 435600 #10 acre in sq feet
dem = 'dem_'+str(select_data)
c_lines = f'v_carve_{select_data}'
accum = f'accum_{select_data}'
drain_dir = f'drain_dir_{select_data}'
r_basins = f'r_basins_{select_data}'
v_basins = f'v_basins_{select_data}'
sub_basins = f'subbasins_{select_data}'
v_streams = f'stream_{select_data}'
r_streams = f'r_streams_{select_data}'
twi = f'r_twi_{select_data}'
v_outlet = f'outlet_{select_data}'

In [51]:
basins+'_buffer'

'aoi_buffer'

In [52]:
dem

'dem_020402020602'

In [53]:
#add mask and reset region
gs.run_command('v.buffer',input=basins,output=basins+'_buffer',distance = buffer)
#mask outside cells
gs.run_command('g.region',vector = basins+'_buffer',align=dem)
gs.run_command('r.mask',vector = basins+'_buffer')

gs.run_command('g.region',raster = dem,align=dem,zoom=dem)
reg = gs.parse_command('g.region',raster = dem,align=dem,zoom=dem,flags='gp')

In [54]:
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

## Set watershed outlet information in map below

In [21]:
import ipyleaflet
from ipyleaflet import Map, Marker
from shapely import Point
center = (aoi.centroid.to_crs(4326).iloc[0].y, aoi.centroid.to_crs(4326).iloc[0].x) 
m = ipyleaflet.Map(center=center, zoom=12)
gj.Raster(r_streams).add_to(m)
gj.Vector(basins).add_to(m)

# Create and place a marker

# Make the marker draggable
marker = Marker(location=center, draggable=True)

# Define a callback function to get coordinates
def on_marker_move(event):
    updated_coords = event['new']
    points = [Point(updated_coords[1],updated_coords[0])]
    # Create a GeoDataFrame and save to shapefile
    gdf = gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")
    gdf.to_file(outlet_point)
# Attach the observer to the 'location' property
marker.observe(on_marker_move, 'location')

m.add_layer(marker)

display(m)

/tmp/ipykernel_725/3005452823.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = (aoi.centroid.to_crs(4326).iloc[0].y, aoi.centroid.to_crs(4326).iloc[0].x)


Map(center=[39.92478710362279, -75.39751360913567], controls=(ZoomControl(options=['position', 'zoom_in_text',…

## Watershed Delineation

In [55]:
#import outlet
gs.run_command('v.import', input= outlet_point,extent='region', output= v_outlet)

In [71]:
gs.run_command('g.region',raster = r_streams)

In [72]:
gs.run_command('r.stream.snap',input = v_outlet, output=f'{v_outlet}_snap',stream_rast = r_streams,radius = 10)

In [73]:
gs.run_command('v.out.ogr',input=f'{v_outlet}_snap',output=vector_dir/f'{v_outlet}_snap.geojson',type= 'point',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_outlet}_snap.geojson').to_file(vector_dir/f'{v_outlet}_snap.shp')

In [74]:
gs.run_command('r.stream.basins',points = f'{v_outlet}_snap', direction=drain_dir,basins=r_basins)

In [75]:
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [76]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'{v_basins}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}.geojson').to_file(vector_dir/f'{v_basins}.shp')

In [77]:
m = gj.InteractiveMap(height=600,width=1200)
m.add_vector(v_outlet)
m.add_vector(f'{v_outlet}_snap')
m.add_vector(v_basins)
m.add_vector(f'v_carve_{select_data}')
m.show()

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## END